# 01 - Import MAD-X and plot optics

Read a MAD-X sequence, calculate periodic Twiss functions, and plot the optics.

In [ ]:
import Pkg
EXAMPLES_DIR = isfile(joinpath(pwd(), "common.jl")) ? pwd() : joinpath(pwd(), "examples")
Pkg.activate(EXAMPLES_DIR)
using CairoMakie, TrackPad

In [ ]:
madx_file = joinpath(EXAMPLES_DIR, "data", "fodo.madx")
ring, beam = read_madx(madx_file; sequence=:FODO, strict=true)
(; elements=length(ring), circumference=total_length(ring), energy_GeV=beam.energy / 1e9)

In [ ]:
tw = periodic_twiss(
    ring, beam; sample_integrator_steps=true, max_step=0.10,
)
chromaticity = getchrom(ring, beam)
(; tune=(tw.tunex, tw.tuney), chromaticity, alphac=tw.alphac)

In [ ]:
set_theme!(Theme(fontsize=14))
fig = Figure(size=(900, 900))
axL = Axis(fig[1, 1], height=90)
plot_lattice!(axL, ring)
hideydecorations!(axL); hidexdecorations!(axL); hidespines!(axL)

axβ = Axis(fig[2, 1], ylabel="β [m]")
lines!(axβ, tw.s, tw.betax; label="βx", linewidth=2.5)
lines!(axβ, tw.s, tw.betay; label="βy", linewidth=2.5)
axislegend(axβ)

axα = Axis(fig[3, 1], ylabel="α")
lines!(axα, tw.s, tw.alphax; label="αx", linewidth=2.5)
lines!(axα, tw.s, tw.alphay; label="αy", linewidth=2.5)
axislegend(axα)

axD = Axis(fig[4, 1], xlabel="s [m]", ylabel="D [m]")
lines!(axD, tw.s, tw.dx; label="Dx", linewidth=2.5)
lines!(axD, tw.s, tw.dy; label="Dy", linewidth=2.5)
axislegend(axD)
linkxaxes!(axL, axβ, axα, axD)
fig